## Section 1 — Library Imports

We import four categories of tools:

1. **Numerical core** (`numpy`): array math, random sampling.
2. **Plotting** (`matplotlib`): to visualize the synthetic data and, later, posteriors.
3. **`Eryn` components**: 
   - `EnsembleSampler`: the MCMC/RJ-MCMC engine that will explore parameter space *inside* each nested sampling iteration.
   - `State`: the container object Eryn uses to hold walker positions, `inds` (on/off masks for variable dimensionality), log-likelihoods, and log-priors simultaneously.
   - `uniform_dist`:" lets us define uniform prior distributions per-parameter, per-branch.
   - `GaussianMove`: a simple in-model proposal (small random-walk steps) used *between* trans-dimensional jumps.
3. **Progress bar** (`tqdm.trange`): nested sampling loops can run thousands of iterations; we want visual feedback.
4. **`corner`**: will be used in Section 10 for posterior corner plots — imported now so all imports live in one place.

We also fix `np.random.seed(...)` so that the synthetic data injection is **reproducible** — a core requirement for debugging Bayesian pipelines (if the data changes every run, you can't tell if a bug is in your sampler or just in stochastic variation).

In [ ]:
# SECTION 1 — LIBRARY IMPORTS

import numpy as np                       # Core array / numerical operations
import matplotlib.pyplot as plt          # Plotting the data, injection, and later posteriors
from tqdm import trange                  # Progress bar for the (potentially long) nested sampling loop

# Eryn imports
from eryn.ensemble import EnsembleSampler    # The main sampler object: handles walkers, temperatures, RJ moves
from eryn.state import State                 # Container object: bundles coordinates + inds + log_like + log_prior
from eryn.prior import uniform_dist          # Defines a uniform prior distribution for one parameter
from eryn.moves import GaussianMove          # A small-step Gaussian random-walk proposal (in-model moves)

import corner                                # For posterior corner plots (used later, in post-processing)

# Reproducibility 
# Fixing the random seed means every time we re-run this notebook from the top,
# we get IDENTICAL synthetic data and identical random draws.
# This is essential for debugging: if our nested sampling evidence estimate
# changes between runs, you want to know it's due to the STOCHASTIC SAMPLER,
# not because the underlying "ground truth" data silently changed too.
np.random.seed(42)

## Section 2 — Toy Model Definition & Synthetic Data Generation

### The physical/statistical model

We model our 1D "signal" as a sum of $k$ Gaussian pulses over a time axis $t$:

$$
s(t) = \sum_{i=1}^{k} A_i \, \exp\!\left(-\frac{(t - \mu_i)^2}{2 \, c_{\text{fixed}}^2}\right)
$$

where, for pulse $i$:
- $A_i$ = **amplitude** (this thesis's Parameter 1 — will be sampled)
- $\mu_i$ = **mean position in time** (Parameter 2 — will be sampled)
- $c_{\text{fixed}}$ = **pulse width** (**fixed for ALL pulses, not sampled**)

### Why fix the width?

Model selection (deciding "how many pulses are present") is already a hard combinatorial + continuous problem. By fixing $c$, we:
1. Reduce the search space dimensionality per pulse (2 instead of 3) so faster RJ-MCMC mixing.
2. Remove a source of **parameter degeneracy**: in the 3-parameter version, a pulse can trade off amplitude vs. width (a tall-narrow pulse can mimic a short-wide one at fixed area), which makes the RJ acceptance ratio harder to reason about. Removing this degeneracy makes it much easier to verify, by eye, whether your Nested Sampling implementation is behaving correctly.

### Synthetic data / injection

We construct a ("injection") with a small number of pulses, add Gaussian measurement noise, and treat the resulting noisy time series as our "observed data." Because we *know* the true number of pulses and their true parameters, we can later check whether the sampler:
- Recovers the correct **posterior on the number of pulses** (model order),
- Recovers unbiased **posteriors on amplitude and mean position**.

In [ ]:
# SECTION 2 - TOY MODEL DIFINITON AND SYNTHETIC DATA GENERATION.

# 2.1 - Single pulse model

pulse_width = 0.1 # This is c in the Gaussin formula. It is identical for every pulse in the model

def single_gaussian_pulse(time_axis, amplitude, mean):
    pulse_values = amplitude * np.exp(-((time_axis - mean) ** 2) / (2 * pulse_width ** 2))
    return pulse_values

# 2.2 - Combine an arbitray number of pulses into one signal
"""
We need to do this because our model can contain multiple pulses simultaneously, 
and we need their total sum. We start with an 'empty canvas' of zeros and use a 
loop to calculate and add each pulse individually on top of the previous ones, 
building the final, unified signal that will be compared to our actual observations.
"""

def combine_all_pulses(time_axis, parameters):
    """
    Sums up an arbitrary number of Gaussian pulses to build the full noiseles
    model signal. This is the function that must handle a VARIABLE number of
    pulses — exactly the quantity our trans-dimensional sampler is trying to infer.

    time_axis : np.ndarray
    list_of_pulse_parameters : iterable of (amplitude, mean) pairs
        Each element describes ONE active pulse. 
    """
    # Example input for 'parameters': [[3.3, -0.2], [2.9, 0.3]] (A list containing 2 pulses)

    # Creates the empty canvas. If time_axis has 500 points, this makes: [0.0, 0.0, ..., 0.0]
    combined_signal = np.zeros_like(time_axis)
    
    # 1st round of loop: takes [3.3, -0.2]
    # 2nd round of loop: takes [2.9, 0.3]
    for single_pulse_parameters in parameters:
        
        # Unpacking the box: e.g., amplitude gets 3.3, mean gets -0.2
        amplitude, mean = single_pulse_parameters
        
        # Draws the pulse using 3.3 and -0.2, then adds it (+) to the existing combined_signal
        combined_signal += single_gaussian_pulse(time_axis, amplitude, mean)
        
    # After the loop finishes, returns the fully constructed waveform containing all pulses
    return combined_signal

# 2.3 Build the time axis
number_of_time_samples = 500
time_axis = np.linspace(-1,1,number_of_time_samples)

# 2.4 - Define the TRUE injected pulses
true_injected_pulse_parameters = [
    [3.3, -0.2],
    [3.4,  0.0],
    [2.9,  0.3],
]

true_number_of_pulses = len(true_injected_pulse_parameters)
noiseless_injected_signal = combine_all_pulses(time_axis, true_injected_pulse_parameters)

# 2.5 Add Gaussian measurement noise to build the "observed data"

